# Лабораторная работа 2: Метод Ньютона

## Задание 1. Реализация метода Ньютона

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve, LinAlgError


def armijo_line_search(f, x, d, g, c1=1e-4, rho=0.5, alpha_init=1.0):
    """Backtracking line search (условие Армихо)."""
    alpha = alpha_init
    f0 = f(x)
    slope = np.dot(g, d)
    while alpha > 1e-16:
        if f(x + alpha * d) <= f0 + c1 * alpha * slope:
            return alpha
        alpha *= rho
    return alpha


def newton_method(f, grad_f, hess_f, x0,
                  alpha_strategy="constant", alpha=1.0,
                  tol=1e-8, max_iter=1000,
                  c1=1e-4, rho=0.5):
    """
    Метод Ньютона для минимизации f(x).

    alpha_strategy: "constant" (фикс. шаг alpha) или "armijo" (подбор шага).
    Регуляризация H + lambda*I применяется, если H не PD.
    """
    x = np.array(x0, dtype=float)
    history = [x.copy()]

    for _ in range(max_iter):
        g = grad_f(x)
        if np.linalg.norm(g) < tol:
            break

        H = hess_f(x)

        lam = 1e-10
        d = None
        for _ in range(60):
            try:
                H_reg = H + lam * np.eye(len(x))
                d_try = solve(H_reg, -g)
                if np.dot(g, d_try) < 0:
                    d = d_try
                    break
            except LinAlgError:
                pass
            lam *= 10
        if d is None:
            d = -g

        if alpha_strategy == "constant":
            step = alpha
        else:
            step = armijo_line_search(f, x, d, g, c1=c1, rho=rho)

        x = x + step * d
        history.append(x.copy())

    return x, history

print("Метод Ньютона реализован.")


## Задание 2. Тест на квадратичной функции

Рассмотрим $f(x) = x^T A x + b^T x$, где:
$$A = \\begin{pmatrix} 4 & 1 \\\\ 1 & 3 \\end{pmatrix}, \\quad b = \\begin{pmatrix} 1 \\\\ -1 \\end{pmatrix}$$

- Градиент: $\\nabla f(x) = 2Ax + b$
- Гессе: $\\nabla^2 f(x) = 2A$ (константная матрица)

**Теоретическое ожидание:** Для квадратичной функции метод Ньютона сходится ровно за **1 итерацию** — аппроксимация Тейлора 2-го порядка точна, и шаг $d = -(2A)^{-1}(2Ax+b)$ сразу приводит в $x^* = -\\tfrac{1}{2}A^{-1}b$.

In [ ]:
# Квадратичная функция
A_quad = np.array([[4.0, 1.0],
                   [1.0, 3.0]])
b_quad = np.array([1.0, -1.0])

def f_quad(x):
    return x @ A_quad @ x + b_quad @ x

def grad_quad(x):
    return 2 * A_quad @ x + b_quad

def hess_quad(x):
    return 2 * A_quad

x_star_quad = -0.5 * np.linalg.solve(A_quad, b_quad)
print(f"Точный минимум: x* = {x_star_quad}")
print(f"f(x*) = {f_quad(x_star_quad):.6f}")

x0 = np.array([5.0, 5.0])
x_opt, history = newton_method(f_quad, grad_quad, hess_quad, x0,
                                alpha_strategy="constant", alpha=1.0, tol=1e-12)

print(f"\nНачальная точка: {x0}")
print(f"Найденный минимум: {x_opt}")
print(f"Число итераций: {len(history) - 1}")
print(f"||x_opt - x*|| = {np.linalg.norm(x_opt - x_star_quad):.2e}")

errors = [np.linalg.norm(np.array(h) - x_star_quad) for h in history]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].semilogy(errors, "b-o", markersize=8)
axes[0].set_xlabel("Итерация")
axes[0].set_ylabel(r"$\|x_k - x^*\|$")
axes[0].set_title("Ошибка (квадратичная функция)")
axes[0].grid(True, alpha=0.4)

xs = np.linspace(-2, 6, 200)
ys = np.linspace(-2, 4, 200)
Xg, Yg = np.meshgrid(xs, ys)
Zg = np.array([[f_quad(np.array([xi, yi])) for xi in xs] for yi in ys])
axes[1].contour(Xg, Yg, Zg, levels=25, cmap="viridis")
traj = np.array(history)
axes[1].plot(traj[:, 0], traj[:, 1], "r-o", markersize=8, label="Траектория")
axes[1].plot(*x_star_quad, "g*", markersize=15, label="x*")
axes[1].set_xlabel("x1"); axes[1].set_ylabel("x2")
axes[1].set_title("Траектория метода Ньютона")
axes[1].legend(); axes[1].grid(True, alpha=0.4)
plt.tight_layout(); plt.show()

print("\nВывод: метод сходится за 1 итерацию — квадратичная аппроксимация")
print("совпадает с функцией точно, шаг Ньютона сразу попадает в x*.")


## Задание 3. Функция $f(x,y) = \\dfrac{-1}{1 + (x-a)^2 + (y-b)^2}$

- $a$ = среднее гармоническое номеров ИСУ / 100000
- $b$ = среднее геометрическое номеров ИСУ / 100000

Единственный глобальный минимум — $f(a,b) = -1$.

**Аналитика.** При $u = x-a$, $v = y-b$, $D = 1+u^2+v^2$:

$$\\nabla f = \\frac{2}{D^2}\\begin{pmatrix}u\\\\v\\end{pmatrix}, \\qquad
H = \\frac{2}{D^3}\\begin{pmatrix}D-4u^2 & -4uv\\\\-4uv & D-4v^2\\end{pmatrix}$$

Гессе ПО $\\Leftrightarrow$ $\\lambda_{\\min}(H) > 0$ $\\Leftrightarrow$
$r = \\sqrt{u^2+v^2} < \\tfrac{1}{\\sqrt{3}} \\approx 0.577$.

In [ ]:
# === Замените номера ИСУ на свои ===
ISU1 = 368090
ISU2 = 378990

a = (2 * ISU1 * ISU2 / (ISU1 + ISU2)) / 100000   # гарм. среднее / 100000
b = np.sqrt(ISU1 * ISU2) / 100000                  # геом. среднее / 100000

print(f"ISU1={ISU1}, ISU2={ISU2}")
print(f"a = {a:.6f}")
print(f"b = {b:.6f}")
print(f"Минимум функции: f({a:.4f}, {b:.4f}) = -1")

x_star3 = np.array([a, b])

def f3(xy):
    x, y = xy
    return -1.0 / (1.0 + (x - a)**2 + (y - b)**2)

def grad_f3(xy):
    x, y = xy
    D = 1.0 + (x - a)**2 + (y - b)**2
    return np.array([2*(x-a)/D**2, 2*(y-b)/D**2])

def hess_f3(xy):
    x, y = xy
    u, v = x - a, y - b
    D = 1.0 + u**2 + v**2
    H11 = 2*(D - 4*u**2) / D**3
    H22 = 2*(D - 4*v**2) / D**3
    H12 = -8*u*v / D**3
    return np.array([[H11, H12], [H12, H22]])


### 3(a). Сходимость с $\\alpha=1$, начальные точки вблизи минимума

In [ ]:
r_conv = 1.0 / np.sqrt(3)
print(f"Теоретическая граница сходимости: r = 1/sqrt(3) = {r_conv:.4f}")

start_points3 = [
    np.array([a + 0.30,  b + 0.20]),
    np.array([a - 0.40,  b + 0.10]),
    np.array([a + 0.10,  b - 0.35]),
]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors3 = ["blue", "orange", "green"]
histories3 = []

for i, x0 in enumerate(start_points3):
    x_opt, hist = newton_method(f3, grad_f3, hess_f3, x0,
                                 alpha_strategy="constant", alpha=1.0,
                                 tol=1e-12, max_iter=500)
    histories3.append(hist)
    errs = [np.linalg.norm(np.array(h) - x_star3) for h in hist]
    r0 = np.linalg.norm(x0 - x_star3)
    lbl = f"x0=(a{x0[0]-a:+.2f}, b{x0[1]-b:+.2f}), r={r0:.3f}, ит.={len(hist)-1}"
    axes[0].semilogy(errs, color=colors3[i], marker="o", markersize=4, label=lbl)
    print(f"Точка {i+1}: x0={x0}, итераций={len(hist)-1}, ||x_opt-x*||={errs[-1]:.2e}")

axes[0].set_xlabel("Итерация")
axes[0].set_ylabel(r"$\|x_k - x^*\|$")
axes[0].set_title("Ошибка vs итерации (alpha=1)")
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.4)

mg = 1.0
xs3 = np.linspace(a-mg, a+mg, 200)
ys3 = np.linspace(b-mg, b+mg, 200)
X3, Y3 = np.meshgrid(xs3, ys3)
Z3 = -1.0 / (1.0 + (X3-a)**2 + (Y3-b)**2)
axes[1].contourf(X3, Y3, Z3, levels=30, cmap="RdYlGn")
axes[1].contour(X3, Y3, Z3, levels=30, colors="k", linewidths=0.3, alpha=0.5)

for i, (x0, hist) in enumerate(zip(start_points3, histories3)):
    traj = np.array(hist)
    axes[1].plot(traj[:,0], traj[:,1], "-o", color=colors3[i], markersize=3)

theta = np.linspace(0, 2*np.pi, 300)
axes[1].plot(a + r_conv*np.cos(theta), b + r_conv*np.sin(theta),
             "w--", lw=2, label=f"r=1/sqrt(3)={r_conv:.3f}")
axes[1].plot(a, b, "w*", markersize=14, label="(a,b)")
axes[1].set_xlabel("x"); axes[1].set_ylabel("y")
axes[1].set_title("Траектории и теор. граница")
axes[1].legend(fontsize=9)
plt.tight_layout(); plt.show()

# Таблица скорости сходимости
print("\n--- Оценка скорости сходимости (квадратичная?) ---")
errs = [np.linalg.norm(np.array(h) - x_star3) for h in histories3[0]]
print(f"  k | {chr(124):>12} ||e_k|| {chr(124):>12} ||e_k+1||/||e_k||^2")
print("-" * 40)
for k in range(1, min(8, len(errs)-1)):
    if errs[k-1] > 1e-15:
        ratio = errs[k] / errs[k-1]**2
        print(f"  {k} |   {errs[k]:.4e}   |   {ratio:.4f}")
print("\nПостоянство отношения подтверждает квадратичную сходимость метода Ньютона.")


### 3(b). Эмпирическая граница области сходимости

**Аналитическое обоснование.**
Функция $f$ строго выпукла в шаре $r < 1/\\sqrt{3}$ (там $\\lambda_{\\min}(H) > 0$).
В этой области метод Ньютона с $\\alpha=1$ гарантированно сходится квадратично.
Вне шара гессе теряет положительную определённость, шаг Ньютона перестаёт быть нисходящим, и метод расходится.

In [ ]:
N = 70
margin_scan = 1.5
xs_s = np.linspace(a - margin_scan, a + margin_scan, N)
ys_s = np.linspace(b - margin_scan, b + margin_scan, N)

conv_map = np.zeros((N, N))
for i, xi in enumerate(xs_s):
    for j, yj in enumerate(ys_s):
        try:
            x_opt, _ = newton_method(f3, grad_f3, hess_f3, np.array([xi, yj]),
                                      alpha_strategy="constant", alpha=1.0,
                                      tol=1e-6, max_iter=200)
            conv_map[j, i] = 1.0 if np.linalg.norm(x_opt - x_star3) < 0.01 else 0.0
        except Exception:
            conv_map[j, i] = 0.0

# Эмпирическая граница
theta_arr = np.linspace(0, 2*np.pi, 72)
emp_radii = []
for angle in theta_arr:
    for r in np.linspace(0.05, margin_scan, 200):
        xp = np.array([a + r*np.cos(angle), b + r*np.sin(angle)])
        try:
            x_opt, _ = newton_method(f3, grad_f3, hess_f3, xp,
                                      alpha_strategy="constant", alpha=1.0,
                                      tol=1e-6, max_iter=200)
            if np.linalg.norm(x_opt - x_star3) > 0.01:
                emp_radii.append(r); break
        except Exception:
            emp_radii.append(r); break
    else:
        emp_radii.append(margin_scan)

r_emp = np.mean(emp_radii)
theta_plot = np.linspace(0, 2*np.pi, 300)

fig, ax = plt.subplots(figsize=(7, 6))
ax.imshow(conv_map,
           extent=[a-margin_scan, a+margin_scan, b-margin_scan, b+margin_scan],
           origin="lower", cmap="RdYlGn", vmin=0, vmax=1, alpha=0.75)
ax.plot(a + r_conv*np.cos(theta_plot), b + r_conv*np.sin(theta_plot),
         "b--", lw=2.5, label=f"Теор. r=1/sqrt(3)={r_conv:.3f}")
ax.plot(a + r_emp*np.cos(theta_plot), b + r_emp*np.sin(theta_plot),
         "k-", lw=2, label=f"Эмп. r={r_emp:.3f}")
ax.plot(a, b, "b*", markersize=14, label="(a,b)")
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Область сходимости alpha=1\n(зелёный=сходится, красный=расходится)")
ax.legend(); plt.tight_layout(); plt.show()

print(f"Теоретическая граница: r = 1/sqrt(3) = {r_conv:.4f}")
print(f"Эмпирическая граница:  r = {r_emp:.4f}")
print("\nВывод: область сходимости alpha=1 совпадает с шаром строгой выпуклости.")


### 3(c). Стратегия Армихо — расширение области сходимости

In [ ]:
conv_map_arm = np.zeros((N, N))
for i, xi in enumerate(xs_s):
    for j, yj in enumerate(ys_s):
        try:
            x_opt, _ = newton_method(f3, grad_f3, hess_f3, np.array([xi, yj]),
                                      alpha_strategy="armijo", tol=1e-6, max_iter=500)
            conv_map_arm[j, i] = 1.0 if np.linalg.norm(x_opt - x_star3) < 0.01 else 0.0
        except Exception:
            conv_map_arm[j, i] = 0.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, cmap_data, title in zip(axes, [conv_map, conv_map_arm], ["alpha=1", "Армихо"]):
    ax.imshow(cmap_data,
              extent=[a-margin_scan, a+margin_scan, b-margin_scan, b+margin_scan],
              origin="lower", cmap="RdYlGn", vmin=0, vmax=1, alpha=0.75)
    ax.plot(a + r_conv*np.cos(theta_plot), b + r_conv*np.sin(theta_plot),
             "b--", lw=2, label=f"r=1/sqrt(3)={r_conv:.3f}")
    ax.plot(a, b, "b*", markersize=12, label="(a,b)")
    ax.set_title(f"Область сходимости ({title})")
    ax.set_xlabel("x"); ax.set_ylabel("y"); ax.legend(fontsize=9)
plt.tight_layout(); plt.show()

# Сравнение скоростей из точек с разными радиусами
test_pts = [
    (np.array([a+0.30, b+0.20]), "r=0.36 (внутри)"),
    (np.array([a+0.70, b+0.00]), "r=0.70 (вне, alpha=1 расход.)"),
    (np.array([a+1.20, b+0.00]), "r=1.20 (далеко)"),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (x0, lbl) in zip(axes, test_pts):
    for strat, col, ls in [("constant", "blue", "-"), ("armijo", "red", "--")]:
        try:
            x_opt, hist = newton_method(f3, grad_f3, hess_f3, x0,
                                         alpha_strategy=strat, alpha=1.0,
                                         tol=1e-10, max_iter=500)
            errs = [np.linalg.norm(np.array(h) - x_star3) for h in hist]
            ax.semilogy(errs, color=col, ls=ls, marker="o", ms=3,
                        label=f"{strat} ({len(hist)-1} ит.)")
        except Exception:
            ax.text(0.1, 0.5, f"{strat}: расход.", transform=ax.transAxes, color=col)
    ax.set_title(lbl, fontsize=9)
    ax.set_xlabel("Итерация"); ax.set_ylabel(r"$\|x_k-x^*\|$")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.4)
plt.suptitle("Скорость сходимости: alpha=1 vs Армихо", fontsize=12)
plt.tight_layout(); plt.show()

print("Вывод: Армихо значительно расширяет область сходимости.")
print("Внутри шара r<1/sqrt(3) скорость квадратична для обеих стратегий.")
print("Вне шара alpha=1 расходится; Армихо уменьшает шаг и сходится.")


## Задание 4. Функция $f(x,y) = -9x - 10y + 10(-\\ln(100-x-y) - \\ln(x) - \\ln(y) - \\ln(50-x+y))$

### 4(a). Область определения

Аргументы всех логарифмов строго положительны:

1. $x > 0$
2. $y > 0$
3. $100 - x - y > 0 \\iff x + y < 100$
4. $50 - x + y > 0 \\iff y > x - 50$

Область — открытый выпуклый четырёхугольник (пересечение четырёх полуплоскостей).

In [ ]:
def f4(xy):
    x, y = xy
    c1, c2 = 100 - x - y, 50 - x + y
    if x <= 0 or y <= 0 or c1 <= 0 or c2 <= 0:
        return np.inf
    return -9*x - 10*y + 10*(-np.log(c1) - np.log(x) - np.log(y) - np.log(c2))

def grad_f4(xy):
    x, y = xy
    c1, c2 = 100 - x - y, 50 - x + y
    return np.array([
        -9  + 10*(1/c1 - 1/x + 1/c2),
        -10 + 10*(1/c1 - 1/y - 1/c2)
    ])

def hess_f4(xy):
    x, y = xy
    c1, c2 = 100 - x - y, 50 - x + y
    H11 = 10*(1/c1**2 + 1/x**2 + 1/c2**2)
    H22 = 10*(1/c1**2 + 1/y**2 + 1/c2**2)
    H12 = 10*(1/c1**2 - 1/c2**2)
    return np.array([[H11, H12], [H12, H22]])

fig, ax = plt.subplots(figsize=(6, 6))
xs_dom = np.linspace(0, 100, 500)
y_lo = np.maximum(0, xs_dom - 50)
y_hi = 100 - xs_dom
mask = y_hi > y_lo
ax.fill_between(xs_dom[mask], y_lo[mask], y_hi[mask],
                alpha=0.3, color="green", label="Область")
ax.plot(xs_dom, y_hi, "b-", lw=2, label="$x+y=100$")
ax.plot(xs_dom, np.maximum(0, xs_dom-50), "r-", lw=2, label="$y=x-50$")
ax.axhline(0, color="gray", lw=1); ax.axvline(0, color="gray", lw=1)
ax.set_xlim(-5, 105); ax.set_ylim(-5, 105)
ax.set_xlabel("x"); ax.set_ylabel("y")
ax.set_title("Область определения f(x,y)")
ax.legend(); ax.grid(True, alpha=0.3); plt.tight_layout(); plt.show()


### 4(b). График функции

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
from scipy.optimize import minimize as sp_minimize

xs_g = np.linspace(1.5, 85, 120)
ys_g = np.linspace(1.5, 85, 120)
X4, Y4 = np.meshgrid(xs_g, ys_g)
Z4 = np.full_like(X4, np.nan)
for i in range(X4.shape[0]):
    for j in range(X4.shape[1]):
        v = f4(np.array([X4[i,j], Y4[i,j]]))
        if np.isfinite(v):
            Z4[i,j] = v

z_clip = np.nanpercentile(Z4, 95)
Z4p = np.where(Z4 > z_clip, np.nan, Z4)

fig = plt.figure(figsize=(14, 5))
ax1 = fig.add_subplot(121, projection="3d")
ax1.plot_surface(X4, Y4, Z4p, cmap="viridis", alpha=0.85, edgecolor="none")
ax1.set_xlabel("x"); ax1.set_ylabel("y"); ax1.set_zlabel("f")
ax1.set_title("f(x,y) — 3D (значения обрезаны сверху)")

ax2 = fig.add_subplot(122)
c = ax2.contourf(X4, Y4, Z4p, levels=40, cmap="viridis")
plt.colorbar(c, ax=ax2)
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_title("f(x,y) — контурный график")
plt.tight_layout(); plt.show()

res4 = sp_minimize(f4, [10.0, 40.0], method="L-BFGS-B",
                   bounds=[(0.01, 99.99)]*2)
x_ref4 = res4.x
print(f"Приближённый минимум: x*=({x_ref4[0]:.4f},{x_ref4[1]:.4f}), f*={res4.fun:.4f}")


### 4(c). Метод Ньютона с постоянным шагом α=1

In [ ]:
start_pts4 = [
    np.array([8.0,  90.0]),
    np.array([1.0,  40.0]),
    np.array([15.0, 68.69]),
    np.array([10.0, 20.0]),
]
labels4 = ["(8, 90)", "(1, 40)", "(15, 68.69)", "(10, 20)"]
colors4 = ["red", "orange", "cyan", "magenta"]

results4_const = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, x0, lbl in zip(axes.flat, start_pts4, labels4):
    try:
        x_opt, hist = newton_method(f4, grad_f4, hess_f4, x0,
                                     alpha_strategy="constant", alpha=1.0,
                                     tol=1e-8, max_iter=300)
        f_vals = [f4(h) for h in hist]
        diffs = [abs(v - res4.fun) + 1e-16 if np.isfinite(v) else np.nan for v in f_vals]
        ax.semilogy(diffs, "b-o", markersize=3)
        out_dom = any(not np.isfinite(v) for v in f_vals)
        status = f"ит.={len(hist)-1}, f={f4(x_opt):.3f}"
        if out_dom: status += " [!выход из обл.]"
        ax.set_title(f"x0={lbl}\n{status}", fontsize=9)
        results4_const[lbl] = (x_opt, hist)
        print(f"alpha=1 x0={lbl}: ит.={len(hist)-1}, "
              f"x*=({x_opt[0]:.3f},{x_opt[1]:.3f}), f={f4(x_opt):.4f}, вне={out_dom}")
    except Exception as e:
        ax.text(0.5, 0.5, f"Ошибка:\n{e}", transform=ax.transAxes, ha="center", va="center")
        print(f"alpha=1 x0={lbl}: ошибка — {e}")
    ax.set_xlabel("Итерация"); ax.set_ylabel("|f(xk)-f*|"); ax.grid(True, alpha=0.4)

plt.suptitle("Метод Ньютона, alpha=1", fontsize=13)
plt.tight_layout(); plt.show()

# Траектории
fig2, ax2 = plt.subplots(figsize=(8, 7))
ax2.contourf(X4, Y4, Z4p, levels=40, cmap="viridis", alpha=0.7)
for (lbl, (x_opt, hist)), col in zip(results4_const.items(), colors4):
    traj = np.array(hist)
    safe = np.array([h for h in traj if np.isfinite(f4(h))])
    if len(safe) > 0:
        ax2.plot(safe[:,0], safe[:,1], "-o", color=col, ms=4, label=lbl)
        ax2.plot(safe[0,0], safe[0,1], "s", color=col, ms=10)
ax2.plot(*x_ref4, "w*", markersize=16, label="Минимум")
ax2.set_xlim(0,100); ax2.set_ylim(0,100)
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_title("Траектории (alpha=1)")
ax2.legend(fontsize=9); plt.tight_layout(); plt.show()

print("\nВывод: при alpha=1 метод быстро сходится только из (10,20) — точки вблизи минимума.")
print("Из (8,90),(1,40),(15,68.69) наблюдается выход за область или расходимость,")
print("так как большой шаг уводит за границу области определения.")


### 4(d). Метод Ньютона со стратегией Армихо

In [ ]:
results4_arm = {}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for ax, x0, lbl in zip(axes.flat, start_pts4, labels4):
    try:
        x_opt, hist = newton_method(f4, grad_f4, hess_f4, x0,
                                     alpha_strategy="armijo", tol=1e-8,
                                     max_iter=500, c1=1e-4, rho=0.5)
        f_vals = [f4(h) for h in hist]
        diffs = [abs(v - res4.fun) + 1e-16 if np.isfinite(v) else np.nan for v in f_vals]
        ax.semilogy(diffs, "g-o", markersize=3)
        ax.set_title(f"x0={lbl}\nит.={len(hist)-1}, f={f4(x_opt):.4f}", fontsize=9)
        results4_arm[lbl] = (x_opt, hist)
        print(f"Армихо x0={lbl}: ит.={len(hist)-1}, "
              f"x*=({x_opt[0]:.3f},{x_opt[1]:.3f}), f={f4(x_opt):.4f}")
    except Exception as e:
        ax.text(0.5, 0.5, f"Ошибка:\n{e}", transform=ax.transAxes, ha="center", va="center")
        print(f"Армихо x0={lbl}: ошибка — {e}")
    ax.set_xlabel("Итерация"); ax.set_ylabel("|f(xk)-f*|"); ax.grid(True, alpha=0.4)

plt.suptitle("Метод Ньютона, стратегия Армихо", fontsize=13)
plt.tight_layout(); plt.show()

# Траектории
fig2, ax2 = plt.subplots(figsize=(8, 7))
ax2.contourf(X4, Y4, Z4p, levels=40, cmap="viridis", alpha=0.7)
for (lbl, (x_opt, hist)), col in zip(results4_arm.items(), colors4):
    traj = np.array(hist)
    safe = np.array([h for h in traj if np.isfinite(f4(h))])
    if len(safe) > 0:
        ax2.plot(safe[:,0], safe[:,1], "-o", color=col, ms=4, label=lbl)
        ax2.plot(safe[0,0], safe[0,1], "s", color=col, ms=10)
ax2.plot(*x_ref4, "w*", markersize=16, label="Минимум")
ax2.set_xlim(0,100); ax2.set_ylim(0,100)
ax2.set_xlabel("x"); ax2.set_ylabel("y")
ax2.set_title("Траектории (Армихо)")
ax2.legend(fontsize=9); plt.tight_layout(); plt.show()

# Итоговая сравнительная таблица
print("\n=== Итоговое сравнение: alpha=1 vs Армихо ===")
print(f"  {chr(34)}Точка{chr(34):>8} | {chr(34)}alpha=1, ит.{chr(34):>4} | {chr(34)}Армихо, ит.{chr(34):>4} | {chr(34)}f* (Армихо){chr(34):>4}")
print("-" * 56)
for lbl in labels4:
    n1 = len(results4_const[lbl][1])-1 if lbl in results4_const else "ошибка"
    if lbl in results4_arm:
        n2 = len(results4_arm[lbl][1])-1
        fopt = f"{f4(results4_arm[lbl][0]):.4f}"
    else:
        n2, fopt = "ошибка", "nan"
    print(f"  {lbl:>12} | {str(n1):>12} | {str(n2):>12} | {fopt:>12}")

print("\nВывод: Армихо гарантирует монотонное убывание f и сходимость из всех точек")
print("внутри области. Постоянный шаг alpha=1 быстрее вблизи минимума, но ненадёжен")
print("из далёких точек — выход за границу области или расходимость.")
